## Παραδοτέο 2

### Εκφώνηση:
Ανακατασκευή του συνόλου των 2 κειμένων με χρήση 3 διαφορετικών αυτόματων βιβλιοθηκών Python pipelines.

---

### Κείμενο 1:
Today is our Dragon Boat Festival, in our Chinese culture, to celebrate it with all safety and greatness in our lives. Hope you too, to enjoy it as my deepest wishes.  
Thank you for your message to show our words to the doctor, as his next contract checking, to all of us.  
I got this message to see the approved message. In fact, I have received the message from the professor, to show me this, a couple of days ago. I greatly appreciate the full support of the professor for our Springer proceedings publication.

---

### Κείμενο 2:
During our final discussion, I told him about the new submission — the one we were waiting for since last autumn, but the updates were confusing as they did not include the full feedback from the reviewer or maybe the editor?  
Anyway, I believe the team, although there was a bit of delay and less communication in recent days, really tried their best for the paper and cooperation. We should be grateful, I mean all of us, for the acceptance and efforts until the Springer link finally came last week, I think.  
Also, kindly remind me, please, if the doctor still plans to edit the acknowledgments section before sending it again. Because I didn’t see that part finalized yet, or maybe I missed it. I apologize if so.  
Overall, let us make sure all are safe and celebrate the outcome with strong coffee and future targets.

---

### Μοντέλα που χρησιμοποιήθηκαν:

1. **Πρώτο μοντέλο: Vamsi/T5_Paraphrase_Paws**  
   [Vamsi/T5_Paraphrase_Paws - Hugging Face](https://huggingface.co/Vamsi/T5_Paraphrase_Paws)  

2. **Δεύτερο μοντέλο: ramsrigouthamg/t5_paraphraser**  
   [ramsrigouthamg/t5_paraphraser - Hugging Face](https://huggingface.co/ramsrigouthamg/t5_paraphraser)  

3. **Τρίτο μοντέλο: prithivida/grammar_error_correcter_v1**  
   [prithivida/grammar_error_correcter_v1 - Hugging Face](https://huggingface.co/prithivida/grammar_error_correcter_v1)  
   Το συγκεκριμένο μοντέλο χρησιμοποιήθηκε για τη διόρθωση γραμματικών και ορθογραφικών λαθών.

4. **Τέταρτο μοντέλο: eugenesiow/bart-paraphrase**  
   [eugenesiow/bart-paraphrase - Hugging Face](https://huggingface.co/eugenesiow/bart-paraphrase)  
   Paraphrasing/reconstruction με αρχιτεκτονική BART.

5. **Πέμπτο μοντέλο: stanford-oval/paraphraser-bart-large**  
   [stanford-oval/paraphraser-bart-large - Hugging Face](https://huggingface.co/stanford-oval/paraphraser-bart-large)  
   Sentence-level paraphrasing με BART, βασισμένο σε back-translation.

6. **Έκτο μοντέλο: EleutherAI/gpt-neo-2.7B**  
   [EleutherAI/gpt-neo-2.7B - Hugging Face](https://huggingface.co/EleutherAI/gpt-neo-2.7B)  
   Causal LM για prompt-based επαναδιατύπωση (few-shot rephrasing).

7. **Έβδομο μοντέλο: redphoenix1996/srs_ambiguity_model**  
   [redphoenix1996/srs_ambiguity_model - Hugging Face](https://huggingface.co/redphoenix1996/srs_ambiguity_model)  
   Μοντέλο ταξινόμησης ασάφειας κειμένου (text-classification).


In [ ]:
"""pip install tiktoken sentencepiece"""
"""!pip install tiktoken sentencepiece huggingface_hub"""
                                                                                            #test code dont run chris
"""from huggingface_hub import login
# Paste in a token you get from https://huggingface.co/settings/tokens
login(token="")  """
#!pip install transformers torch scikit-learn


In [7]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, AutoModelForSequenceClassification
import torch

# Models to run
models = [
    "Vamsi/T5_Paraphrase_Paws",
    "ramsrigouthamg/t5_paraphraser",
    "prithivida/grammar_error_correcter_v1",
    "eugenesiow/bart-paraphrase",
    "stanford-oval/paraphraser-bart-large",
    #"EleutherAI/gpt-neo-2.7B",                            #takes ages to run like 15 min ages 
    #"redphoenix1996/srs_ambiguity_model"                  #private repo remember to delete 
]

texts = {
    "text1": (
        "Today is our dragon boat festival, in our Chinese culture, "
        "to celebrate it with all safe and great in our lives. "
        "Hope you too, to enjoy it as my deepest wishes. "
        "Thank your message to show our words to the doctor, as his next contract checking, to all of us. "
        "I got this message to see the approved message. In fact, I have received the message from the "
        "professor, to show me this a couple of days ago. I am very appreciated the full support of the "
        "professor, for our Springer proceedings publication."
    ),
    "text2": (
        "During our final discuss, I told him about the new submission — the one we were waiting since "
        "last autumn, but the updates was confusing as it not included the full feedback from reviewer or "
        "maybe editor? Anyway, I believe the team, although bit delay and less communication at recent days, they really "
        "tried best for paper and cooperation. We should be grateful, I mean all of us, for the acceptance "
        "and efforts until the Springer link came finally last week, I think. "
        "Also, kindly remind me please, if the doctor still plan for the acknowledgments section edit before "
        "he sending again. Because I didn’t see that part final yet, or maybe I missed, I apologize if so. "
        "Overall, let us make sure all are safe and celebrate the outcome with strong coffee and future targets."
    )
}

device = 0 if torch.cuda.is_available() else -1

for text_name, text in texts.items():
    print(f"\n\n######## {text_name.upper()} ########\n")
    for model_name in models:
        print(f"-- Model: {model_name} --")
        
        if "gpt-neo" in model_name:
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model     = AutoModelForCausalLM.from_pretrained(model_name)
            paraphraser = pipeline(
                "text-generation", model=model, tokenizer=tokenizer, device=device
            )
            prompt = f"Rephrase clearly: \"{text}\""
            outputs = paraphraser(
                prompt,
                max_length=len(tokenizer(prompt)["input_ids"]) + 200,
                num_return_sequences=3,
                do_sample=True,
                top_k=50,
                top_p=0.95
            )
            variants = [out['generated_text'][len(prompt):].strip() for out in outputs]

        elif "srs_ambiguity_model" in model_name:
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model     = AutoModelForSequenceClassification.from_pretrained(model_name)
            classifier = pipeline(
                "text-classification", model=model, tokenizer=tokenizer, device=device
            )
            preds = classifier(text)
            print("Ambiguity classification:", preds, "\n")
            continue

        else:
            tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
            model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)
            paraphraser = pipeline(
                "text2text-generation", model=model, tokenizer=tokenizer, device=device
            )
            outputs = paraphraser(
                text,
                max_length=512,
                num_return_sequences=3,
                do_sample=True,
                top_k=50,
                top_p=0.95
            )
            variants = [out['generated_text'] for out in outputs]

        for i, variant in enumerate(variants, start=1):
            print(f"Variant {i}:\n{variant}\n")




######## TEXT1 ########

-- Model: Vamsi/T5_Paraphrase_Paws --


Device set to use cpu


Variant 1:
Today is our Dragon Boat Festival in our Chinese culture to celebrate it with all safe and great in our lives and hope you as our deepest wishes to enjoy it as well I received this message from Professor to show me this a couple of days ago. I am very grateful for the support from the professor for the publication of our Springer proceedings .

Variant 2:
Today is our dragon boat festival in our Chinese culture to celebrate it with all safe and great in our lives. Hope you too, to enjoy it as my deepest wishes . I received this message to see the approved message from the professor, a couple of days ago , and am very grateful to the full support from the professor, for our Springer proceedings publication .

Variant 3:
Today is our dragon boat festival in our Chinese culture to celebrate with all safe and great in our lives. Hope you enjoy it as my deepest wishes. Thank your message to show the Doctor our words to show him with his next contract check.

-- Model: ramsrigouth

Device set to use cpu


Variant 1:
Today is our dragon boat festival, in our Chinese culture, to celebrate it with all safe and great in our lives. Hope you too, to enjoy it as my deepest wishes. Thank you to see the approved message. In fact, I have received the message from the professor, to show me this a couple of days ago.

Variant 2:
Today is our dragon boat festival, in our Chinese culture, to celebrate it with all safe and great in our lives. Hope you too, to enjoy it as my deepest wishes. I got this message to see the approved message. In fact, I have received the message from the professor, to show me this a couple of days ago. I am very appreciated the full support of the professor, for our Springer proceedings publication.

Variant 3:
I got this message to see our words to the doctor, as his next contract checking, to all of us. Hope you too, to enjoy it with all safe and great in our lives. Thank your message to show our words to the doctor as his next contract checking, to all of us. I'm very ap

...

Variant 1:
Finally, I said to Mr. Kuhlman about the new submission — the one we were waiting for since last autumn, but the updates were confusing as it did not include the full feedback from reviewers, or maybe editor? Anyway, I think the team, although we didn’t see that part

Variant 2:
Overall, let us make sure all are safe and celebrate the outcome with strong coffee and future targets. We will keep you posted to see the latest reprint — the one we were waiting for since last autumn, but the updates are confusing, as they were not included with the full feedback from reviewers and

Variant 3:
After our final discussion, I told him about the new submission — the one we were waiting for since last autumn, but the updates were confusing as it did not include the full feedback from reviewer or maybe editor? Anyway, I believe the team, although a bit of delay and less communication of late,

-- Model: eugenesiow/bart-paraphrase --


Device set to use cpu


Variant 1:
During our final discuss, I told him about the new submission -- the one we were waiting since last autumn, but the updates was confusing as it not included the full feedback from the reviewer or maybe editor. We should be grateful, I mean all of us, if the doctor still plan for the acknowledgments section edit before he sends again. Because I didn’t see that part final yet, or maybe I missed, I apologize if so.

Variant 2:
During our final discuss, I told him about the new submission -- the one we were waiting since last autumn, but the updates was confusing as it not included the full feedback from reviewer or maybe editor. We should be grateful, I mean all of us, for the acceptance and efforts until the Springer link came finally last week, I apologize if so. Also, kindly remind me please, if the doctor still plan for the acknowledgments section edit before he sending again, I believe the team, although bit delay and less communication at recent days.

Variant 3:
During o

Device set to use cpu


Variant 1:
 When I was finalizing the talk last week, I believe we should all be grateful, we should all know that she made a new submission last year, too, but the update was confusing, because it did not yet include the full feedback of a reviewer or one of her editors, and it could have gotten a bit muddled during the last few days, so if that is the case, please forgive me.

Variant 2:
 As we ended our last talk, I think we should also thank the team for accepting and making a last push at last last week, as it had gotten confusing with the missing final from the reviewer or from someone's drafts, and I apologize if we overreacted here, let's enjoy the finish of our meal and drinks, together with all the goals.

Variant 3:
 And we should all, I think, be grateful to the team for accepting and eventually submitting that new submission last week, the one we've been waiting for since last fall, but the update was puzzling, because it didn't contain all the feedback from the reviewers 

## Αποτελέσματα

---

### Κείμενο 1:

Today is our Dragon Boat Festival, in our Chinese culture, to celebrate it with all safety and greatness in our lives. Hope you too, to enjoy it as my deepest wishes.  
Thank you for your message to show our words to the doctor, as his next contract checking, to all of us.  
I got this message to see the approved message. In fact, I have received the message from the professor, to show me this, a couple of days ago. I greatly appreciate the full support of the professor for our Springer proceedings publication.

---

#### Μοντέλο: Vamsi/T5_Paraphrase_Paws

- **Variant 1:**  
  Today is our Dragon Boat Festival in our Chinese culture to celebrate it with all safe and great in our lives and hope you as our deepest wishes to enjoy it as well I received this message from Professor to show me this a couple of days ago. I am very grateful for the support from the professor for the publication of our Springer proceedings .

- **Variant 2:**  
  Today is our dragon boat festival in our Chinese culture to celebrate it with all safe and great in our lives. Hope you too, to enjoy it as my deepest wishes . I received this message to see the approved message from the professor, a couple of days ago , and am very grateful to the full support from the professor, for our Springer proceedings publication .

- **Variant 3:**  
  Today is our dragon boat festival in our Chinese culture to celebrate with all safe and great in our lives. Hope you enjoy it as my deepest wishes. Thank your message to show the Doctor our words to show him with his next contract check.

---

#### Μοντέλο: ramsrigouthamg/t5_paraphraser

- **Variant 1:**  
  Today is our dragon boat festival, in our Chinese culture, to celebrate it with all safe and great in our lives. Hope you too, to enjoy it as my deepest wishes. Thank you to see the approved message. In fact, I have received the message from the professor, to show me this a couple of days ago.

- **Variant 2:**  
  Today is our dragon boat festival, in our Chinese culture, to celebrate it with all safe and great in our lives. Hope you too, to enjoy it as my deepest wishes. I got this message to see the approved message. In fact, I have received the message from the professor, to show me this a couple of days ago. I am very appreciated the full support of the professor, for our Springer proceedings publication.

- **Variant 3:**  
  I got this message to see our words to the doctor, as his next contract checking, to all of us. Hope you too, to enjoy it with all safe and great in our lives. Thank your message to show our words to the doctor as his next contract checking, to all of us. I have received this message to see the approved message, in fact, I have received the message from the professor, to show me this, a couple of days ago.

---

#### Μοντέλο: prithivida/grammar_error_correcter_v1

- **Variant 1:**  
  Today is our dragon boat festival, in our Chinese culture, to celebrate it with all safe and great in our lives. Hope you too, to enjoy it as my deepest wishes. Thank your message to the doctor, with his next contract checking, to all of us. I got this message to see the approved

- **Variant 2:**  
  Today is our dragon boat festival, in our Chinese culture, to celebrate it with all safety and great in our lives. Hope you too, to celebrate it with all that is safe and great in our lives. I got the message to show our words to the doctor, as his next contract checking, to all of

- **Variant 3:**  
  Today is our dragon boat festival, in our Chinese culture, to celebrate it with all safety and great in our lives. Thanks for your message to show our words to the doctor, and on his next contract checking, to all of us. I have received the message to see the approved message. I am very appreciated the

---

#### Μοντέλο: eugenesiow/bart-paraphrase

- **Variant 1:**  
  Today is our dragon boat festival, in our Chinese culture, to celebrate it with all safe and great in our lives. Thank your message to show our words to the doctor, as his next contract checking, to all of us. I am very appreciated the full support of the professor, for our Springer proceedings publication.

- **Variant 2:**  
  Today is our dragon boat festival, in our Chinese culture, to celebrate it with all safe and great in our lives. Thank your message to show our words to the doctor as his next contract checking, to all of us. I am very appreciated the full support of the professor, for our Springer proceedings publication.

- **Variant 3:**  
  Today is our dragon boat festival, in our Chinese culture to celebrate it with all safe and great in our lives. Thank your message to show our words to the doctor, as his next contract checking, to all of us. I am very appreciated the full support of the professor, for our Springer proceedings publication.

---

#### Μοντέλο: stanford-oval/paraphraser-bart-large

- **Variant 1:**  
  And I actually received that report a few days ago, from the professor, the full support of the faculty, to celebrate, safely and happily, at the Springer Convention, as I wish we all did. I hope you will take a moment to appreciate this report, from the doctor, to see what's happening.

- **Variant 2:**  
  Actually, I got a message a few days ago from the Professor, to celebrate with all the safe and lovely people in our lives in our life. and I hope you will all also enjoy it, - Thank you for your message. for showing us our words to the doctor as your next screening of our contract.

- **Variant 3:**  
  Thank you for your message that the professor is coming to see me for the festival today of our dragon boat in Chinese culture, to celebrate with everyone safe and well in our lives, and I hope you all enjoy as much as I do.

---

### Κείμενο 2:

During our final discussion, I told him about the new submission — the one we were waiting for since last autumn, but the updates were confusing as they did not include the full feedback from the reviewer or maybe the editor?  
Anyway, I believe the team, although there was a bit of delay and less communication in recent days, really tried their best for the paper and cooperation. We should be grateful, I mean all of us, for the acceptance and efforts until the Springer link finally came last week, I think.  
Also, kindly remind me, please, if the doctor still plans to edit the acknowledgments section before sending it again. Because I didn’t see that part finalized yet, or maybe I missed it. I apologize if so.  
Overall, let us make sure all are safe and celebrate the outcome with strong coffee and future targets.

---

#### Μοντέλο: Vamsi/T5_Paraphrase_Paws

- **Variant 1:**  
  I told him about the new submission during the final discussion — the one we were waiting for since last fall, but the updates were confusing as it was not the full feedback from reviewer or maybe editor? Anyway, I think the team really tried best for paper and cooperation , I should be grateful, I mean all of us if the doctor still plans for the acknowledgments section edit before he sends again, because I didn’t see that part final yet or maybe I missed, I apologize if so .

- **Variant 2:**  
  I told him about the new submission during our final discuss , which we had been waiting for since last autumn, but the updates was confusing because it did not include the full feedback from reviewer or maybe the editor? Anyway, we should be grateful to all of us for the acceptance and efforts until the Springer Link came finally last week I think . Also , kindly remind me that I did not see the final part yet or maybe I missed it ; let us all celebrate the outcome with strong coffee and future goals .

- **Variant 3:**  
  At the end of our discussion, I told him about the new submission — one we had been waiting for since autumn of last year , but the updates was confusing as it did not include the full feedback from reviewer or maybe editor? Anyway, I believe the team used to best for paper and cooperation . We should be grateful, I mean all of us , if the doctor still plan for the acknowledgements section edit before sending again, maybe if I didn’t see that part yet .

---

#### Μοντέλο: ramsrigouthamg/t5_paraphraser

- **Variant 1:**  
  I told him about the new submission — the one we were waiting since last autumn, but the updates was confusing as it not included the full feedback from reviewer or maybe editor? Anyway, I believe the team, although bit delay and less communication at recent days, they really tried best for paper and cooperation. We should be grateful, I mean all of us, for acceptance and efforts until the Springer link came finally last week, I think.

- **Variant 2:**  
  During our final discuss, I told him about the new submission — the one we were waiting since last autumn, but the updates was confusing as it not included the full feedback from reviewer or maybe editor? Anyway, I believe the team, although bit delay and less communication at recent days, they really tried best for paper and cooperation. We should be grateful, I mean all of us, for the acceptance and efforts until the Springer link came finally last week, I think. Also, kindly remind me please, if the doctor still plan for

- **Variant 3:**  
  I want to thank the Springer team for their cooperation, even when I wasn't aware yet the end deadline. We make sure all are safe.

---

#### Μοντέλο: prithivida/grammar_error_correcter_v1

- **Variant 1:**  
  Finally, I said to Mr. Kuhlman about the new submission — the one we were waiting for since last autumn, but the updates were confusing as it did not include the full feedback from reviewers, or maybe editor? Anyway, I think the team, although we didn’t see that part

- **Variant 2:**  
  Overall, let us make sure all are safe and celebrate the outcome with strong coffee and future targets. We will keep you posted to see the latest reprint — the one we were waiting for since last autumn, but the updates are confusing, as they were not included with the full feedback from reviewers and

- **Variant 3:**  
  After our final discussion, I told him about the new submission — the one we were waiting for since last autumn, but the updates were confusing as it did not include the full feedback from reviewer or maybe editor? Anyway, I believe the team, although a bit of delay and less communication of late,

---

#### Μοντέλο: eugenesiow/bart-paraphrase

- **Variant 1:**  
  During our final discuss, I told him about the new submission -- the one we were waiting since last autumn, but the updates was confusing as it not included the full feedback from the reviewer or maybe editor. We should be grateful, I mean all of us, if the doctor still plan for the acknowledgments section edit before he sends again. Because I didn’t see that part final yet, or maybe I missed, I apologize if so.

- **Variant 2:**  
  During our final discuss, I told him about the new submission -- the one we were waiting since last autumn, but the updates was confusing as it not included the full feedback from reviewer or maybe editor. We should be grateful, I mean all of us, for the acceptance and efforts until the Springer link came finally last week, I apologize if so. Also, kindly remind me please, if the doctor still plan for the acknowledgments section edit before he sending again, I believe the team, although bit delay and less communication at recent days.

- **Variant 3:**  
  During our final discuss, I told him about the new submission — the one we were waiting since last autumn, but the updates was confusing as it not included the full feedback from the reviewer or maybe editor. We should be grateful, I mean all of us, for the acceptance and efforts until the Springer link came finally last week, I apologize if so. Also, kindly remind me please, if the doctor still plan for the acknowledgments section edit before he sending again, I believe the team, although bit delay and less communication at recent days.

---

#### Μοντέλο: stanford-oval/paraphraser-bart-large

- **Variant 1:**  
  When I was finalizing the talk last week, I believe we should all be grateful, we should all know that she made a new submission last year, too, but the update was confusing, because it did not yet include the full feedback of a reviewer or one of her editors, and it could have gotten a bit muddled during the last few days, so if that is the case, please forgive me.

- **Variant 2:**  
  As we ended our last talk, I think we should also thank the team for accepting and making a last push at last last week, as it had gotten confusing with the missing final from the reviewer or from someone's drafts, and I apologize if we overreacted here, let's enjoy the finish of our meal and drinks, together with all the goals.

- **Variant 3:**  
  And we should all, I think, be grateful to the team for accepting and eventually submitting that new submission last week, the one we've been waiting for since last fall, but the update was puzzling, because it didn't contain all the feedback from the reviewers or from a potential editor, and I apologize if it was just timing and timing.
